In [18]:
import pandas as pd

In [19]:
# LSS
n_lss_df = pd.read_csv("N_LSS_results.csv")
l_lss_df = pd.read_csv("L_LSS_results.csv")
m_lss_df = pd.read_csv("M_LSS_results.csv")

# IMG
lm_img_df = pd.read_csv("LM_IMG_results.csv")
n_img_df = pd.read_csv("N_IMG_results.csv")

In [20]:
# Mapping column names to their pretty version
COLUMN_NAMES = {
    'nd_filter': 'ND Filter',
    'slit': 'Slit',
    'filter': 'Filter',
    'detmode': 'Detector Mode',
    'dit': 'DIT [s]',
    'fill_frac': '% Filled',
    'mindit_triggered': 'MINDIT Hit',
    'attempted_dit': 'Attempted DIT [s]',
    'mindit': 'MINDIT [s]',
}

# Can adjust the decimal places used in the table
ROUNDING = {
    'DIT [s]': 3,
    '% Filled': 0,
    'Attempted DIT [s]': 3,
    'MINDIT [s]': 3,
}

# When using LSS mode, the slit varies, but with IMG the filter varies
SELECTOR_COLS = {'LSS': 'Slit', 'IMG': 'Filter'}

In [21]:
# I was saving the fill factors as astropy quantities, so im changing them
# so they can be rounded more easily
def to_percent_float(x):
    if hasattr(x, 'value'):          # still a Quantity object
        return float(x.value)
    if isinstance(x, str):           # came back from CSV as text
        return float(x.replace('%', '').strip())
    return float(x)                  # already a plain number

In [ ]:
def format_table(df, mode, include_detmode=False):
    """
    mode: 'LSS' or 'IMG'
    include_detmode: add a 'Detector Mode' column after the selector column
    """
    df = df.copy()

    # strip units to plain floats first
    df['fill_frac'] = df['fill_frac'].apply(to_percent_float)

    cols = ['ND Filter', SELECTOR_COLS[mode]]
    if include_detmode:
        cols.append('Detector Mode')
    cols += ['DIT [s]', '% Filled', 'MINDIT Hit', 'Attempted DIT [s]']

    df = df.rename(columns=COLUMN_NAMES)[cols]

    df = df.round({k: v for k, v in ROUNDING.items() if k in df.columns})

    return df.fillna('-')

In [23]:
# LSS
formatted_l_lss = format_table(l_lss_df, 'LSS')
formatted_m_lss = format_table(m_lss_df, 'LSS')
formatted_n_lss = format_table(n_lss_df, 'LSS', include_detmode=True)

In [24]:
with open("WCU_LSS_L_BB.md", "w") as f:
    f.write(formatted_l_lss.to_markdown(index=False))
with open("WCU_LSS_M_BB.md", "w") as f:
    f.write(formatted_m_lss.to_markdown(index=False))
with open("WCU_LSS_N_BB.md", "w") as f:
    f.write(formatted_n_lss.to_markdown(index=False))

In [25]:
# IMG
formatted_lm_img = format_table(lm_img_df, 'IMG')
formatted_n_img = format_table(n_img_df, 'IMG', include_detmode=True)

In [26]:
with open("WCU_IMG_LM_BB.md", "w") as f:
    f.write(formatted_lm_img.to_markdown(index=False))
with open("WCU_IMG_N_BB.md", "w") as f:
    f.write(formatted_n_img.to_markdown(index=False))